In [22]:
import numpy as np
import matplotlib.pyplot as plt
import gzip

# Data Pre-processing

In [23]:
with gzip.open('./data/MNIST/raw/t10k-images-idx3-ubyte.gz') as file:
    print(np.frombuffer(file.read(64)).shape)

(8,)


In [24]:
def read_idx_images(filepath):
    with gzip.open(filepath, 'rb') as f:

        header = np.frombuffer(f.read(16), dtype='>i4')  
        magic, num_images, rows, cols = header


        data = np.frombuffer(f.read(), dtype=np.uint8)
        images = data.reshape(num_images, rows, cols)
    return images

def read_idx_labels(filepath):
    with gzip.open(filepath, 'rb') as f:

        header = np.frombuffer(f.read(8), dtype='>i4')
        magic, num_labels = header

        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

In [25]:
train_images = read_idx_images("data/MNIST/raw/train-images-idx3-ubyte.gz")
train_labels = read_idx_labels("data/MNIST/raw/train-labels-idx1-ubyte.gz")

test_images = read_idx_images("data/MNIST/raw/t10k-images-idx3-ubyte.gz")
test_labels = read_idx_labels("data/MNIST/raw/t10k-labels-idx1-ubyte.gz")

In [26]:
print(train_images.shape)
print(train_labels.shape)

print(test_images.shape)
print(test_labels.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [27]:
train_images = train_images / np.max(train_images)
test_images = test_images / np.max(test_images)

In [28]:
mean = np.mean(train_images)
std = np.std(train_images)

In [29]:
print(mean)
print(std)

0.1306604762738429
0.30810780385646264


In [30]:
train_data = (train_images - mean) / std
test_data = (test_images - mean) / std

# Model declaration

In [31]:
def ReLU(x):
    c = x.copy()
    c[c < 0] = 0
    return c

In [32]:
class MNIST:
    def __init__(self):
        self.W1 = np.random.rand(64, 784) * 0.01
        self.b1 = np.random.rand(1, 64) * 0.01
        self.W2 = np.random.rand(10, 64) * 0.01
        self.b2 = np.random.rand(1, 10) * 0.01
        self.loss = 0.0

    def forward(self, x, p):

        z1 = x @ self.W1.T + self.b1 # B x 784 @ 784 x 64 + 1 x 64
        a1 = ReLU(z1) # B x 64
        z2 = a1 @ self.W2.T + self.b2 # B x 64 @ 64 x 10 + 1 x 10
        z2 = z2 - np.max(z2, axis=1, keepdims=True)
        q = np.exp(z2) / np.sum(np.exp(z2), axis=1, keepdims=True)


        self.loss = -np.sum(p * np.log(q), axis=0, keepdims=True, dtype=np.float128)

        dz = z1.copy()
        dz[dz < 0] = 0
        dz[dz > 0] = 1

        delta = ((q - p) @ self.W2 * dz) # 64 x 10 @ 10 x 64 * 64 x 64 = 64 x 64
        dW1 = (delta.T @ x) / 64 # 64 x 64 @ 64 x 784 = 64 x 784
        db1 = (np.sum(delta, axis=0, keepdims=True)) / 64 # 1 x 64

        dW2 = ((q - p).T @ a1) / 64 # 10 x 64 @ 64 x 64 = 10 x 64
        db2 = (np.sum(q - p, axis=0, keepdims=True)) / 64 # 1 x 10

        self.W1 = self.W1 - 0.01 * dW1 
        self.b1 = self.b1 - 0.01 * db1 
        self.W2 = self.W2 - 0.01 * dW2 
        self.b2 = self.b2 - 0.01 * db2

    def predict(self, data):
        z1 = data @ self.W1.T + self.b1
        a1 = ReLU(z1)
        z2 = a1 @ self.W2.T + self.b2

        z2 = z2 - np.max(z2, axis=1, keepdims=True)

        return np.exp(z2) / np.sum(np.exp(z2), axis=1, keepdims=True)


In [33]:
def train(model, train_data, label):
    batch_idx = np.random.permutation(60000)
    start = 0
    loss = 0.0
    nb_classes = 10
    for end in range(64, 60000, 64):
        p = np.eye(nb_classes, dtype=np.float64)[label[batch_idx[start:end]]]
        model.forward(train_data[batch_idx[start: end]].reshape(64, 784), p)
        start = end
        loss += model.loss
    loss = loss / 60000

    return model, np.average(loss)

In [34]:
batch_idx = np.random.permutation(60000)
nb_classes = 10
p = np.eye(nb_classes)[train_labels[batch_idx[0:5]]]
print(p)

train_labels[batch_idx[0:5]]

[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]


array([0, 6, 7, 1, 4], dtype=uint8)

In [35]:
model = MNIST()

In [36]:
num_epoch = 5

In [37]:
for epoch in range(num_epoch):
    model, loss = train(model, train_data, train_labels)
    print(loss)

0.1354772154454079766
0.039888774133728824462
0.031757707958837177594
0.028276192869314209504
0.026058729050950862695


In [39]:
import torchmetrics
import torch

In [40]:
test_data = np.asarray(test_data, dtype=np.float32)
test_data = test_data.reshape(10000, 784)
p = torch.from_numpy(test_labels)

In [41]:
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10, average="macro")
recall = torchmetrics.Recall(task="multiclass", num_classes=10, average="macro")
precision = torchmetrics.Precision(task="multiclass", num_classes=10, average="macro")
f1 = torchmetrics.F1Score(task="multiclass", num_classes=10, average="macro")

In [43]:
preds = model.predict(test_data)
preds = torch.from_numpy(preds)

In [44]:
print(f"Accuracy {accuracy(preds, p).item() * 100}")
print(f"Recall {recall(preds, p).item() * 100}")
print(f"Precision {precision(preds, p).item() * 100}")
print(f"F1 Score {f1(preds, p).item() * 100}")

Accuracy 92.97815561294556
Recall 92.97815561294556
Precision 93.01216006278992
F1 Score 92.9801881313324
